In [7]:
from pathlib import Path

import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import kagglehub

In [3]:
SEED = 42
TARGET_COL = 'addicted_label'
ID_COL = 'id'

FAST_DEV_RUN = True
N_SPLITS = 3 if FAST_DEV_RUN else 5
MAX_EPOCHS = 3 if FAST_DEV_RUN else 25
PATIENCE = 2 if FAST_DEV_RUN else 5
PERM_SAMPLE_SIZE = 2_000 if FAST_DEV_RUN else 25_000
PERM_REPEATS = 1 if FAST_DEV_RUN else 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 4096 if DEVICE.type == 'cuda' else 2048
PRED_BATCH_SIZE = 8192 if DEVICE.type == 'cuda' else 4096
NUM_WORKERS = 0

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print('PyTorch version :', torch.__version__)
print('Device          :', DEVICE)
print('5-fold CV       :', N_SPLITS)
print('Max epochs      :', MAX_EPOCHS)
print('Permutation rows:', f'{PERM_SAMPLE_SIZE:,} / fold')


PyTorch version : 2.13.0+cu130
Device          : cpu
5-fold CV       : 3
Max epochs      : 3
Permutation rows: 2,000 / fold


In [5]:
DATA_DIR = Path(kagglehub.competition_download('playground-series-s6e8'))
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

assert train.drop(columns=TARGET_COL).columns.to_list() == test.columns.to_list()
print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Target rate: {train[TARGET_COL].mean():.4f}')

Train: (691369, 14) | Test: (296302, 13)
Target rate: 0.7094


In [8]:
NUMERIC_COLUMNS = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time',
]
CATEGORICAL_COLUMNS = ['gender', 'stress_level', 'academic_work_impact']

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Create target-free features; safe to apply before CV."""
    X = frame.drop(columns=[TARGET_COL, ID_COL], errors='ignore').copy()
    raw_columns = X.columns.tolist()

    # Preserve missingness explicitly; native tree models also receive numeric NaNs.
    for column in raw_columns:
        X[f'{column}_is_missing'] = X[column].isna().astype('int8')
    X['missing_count'] = X[raw_columns].isna().sum(axis=1).astype('int8')

    # CatBoost needs categorical missing values represented as strings.
    for column in CATEGORICAL_COLUMNS:
        X[column] = X[column].astype('string').fillna('Missing')

    social, gaming = X['social_media_hours'], X['gaming_hours']
    daily, weekend = X['daily_screen_time_hours'], X['weekend_screen_time']
    work_study, sleep = X['work_study_hours'], X['sleep_hours']
    X['social_gaming_hours'] = social + gaming
    X['weekday_weekend_screen_difference'] = weekend - daily
    X['weekend_daily_screen_ratio'] = weekend / (daily + 0.25)
    X['usage_hours_total'] = daily + social + gaming + work_study
    X['screen_to_sleep_ratio'] = daily / (sleep + 0.25)

    for column in ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                   'work_study_hours', 'weekend_screen_time']:
        X[f'{column}_squared'] = X[column] ** 2
    return X

X_train = build_features(train)
X_test = build_features(test)
y = train[TARGET_COL].astype('int8')
assert X_train.columns.tolist() == X_test.columns.tolist()
print(f'Model features: {X_train.shape[1]}')

Model features: 35


In [ ]:
class FoldPreProcessor:

    def __init__(self, num_columns, cat_columns):
        self.num_columns = num_columns
        self.cat_columns = cat_columns

    def fit(self, df: pd.DataFrame):
        